In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import xarray as xr

In [2]:
# ---------------- Paths ----------------

# base_dir = Path("/inputs")
met_dir = Path("/inputs/data_updated_2/time_series")
area_file= Path("/inputs/data_updated_2/attributes/attributes_other.csv")
attributes_files= Path("/inputs/data_updated_2/attributes")

# Define output structure required by neuralhydrology
data_dir = Path("./filtered_data_2")
time_series_dir = data_dir / "time_series"
attributes_dir = data_dir / "attributes"

# Create output directories
data_dir.mkdir(exist_ok=True)
time_series_dir.mkdir(exist_ok=True)
attributes_dir.mkdir(exist_ok=True)

#Streamflow data files
train_dir = "./processed_data_2/highqual_with_short_estimated/train_data.csv"
validation_dir = "./processed_data_2/highqual_with_short_estimated/validation_data.csv"
test_dir = "./processed_data_2/highqual_with_short_estimated/test_data.csv"

In [3]:
# Define variable mapping from CAMELS_UY names to neuralhydrology names
variable_mapping = {
    'temperature_2m_min': 'tmin_C',
    'temperature_2m_max': 'tmax_C',
    'surface_net_solar_radiation_mean': 'srad_W_m2',
    'total_precipitation_sum': 'prcp_mm_day'
}

# Read streamflow data
train_data = pd.read_csv(train_dir, index_col=0, parse_dates=True)
val_data = pd.read_csv(validation_dir, index_col=0, parse_dates=True)
test_data = pd.read_csv(test_dir, index_col=0, parse_dates=True)

# Combine all streamflow data
streamflow_data = pd.concat([train_data, val_data, test_data]).sort_index()

# Get list of basins
basins = streamflow_data.columns.tolist()

basins_mapping = {
    'Paso Mazangano': 'CAMELS_UY_10',
    'Paso de Coelho': 'CAMELS_UY_7',
    'Sarandi del Yi': 'CAMELS_UY_12',
    'Paso de las Toscas': 'CAMELS_UY_8',
    'Paso de las Piedras (R3)': 'CAMELS_UY_15',
    'Paso del Borracho': 'CAMELS_UY_6',
    'Bequelo': 'CAMELS_UY_16',
    'Paso de las Piedras': 'CAMELS_UY_2',
    'Paso Baltasar': 'CAMELS_UY_5',
    'Fraile Muerto': 'CAMELS_UY_11',
    'Paso de los Mellizos': 'CAMELS_UY_14',
    'Paso Manuel Diaz': 'CAMELS_UY_3',
    'Paso Aguiar': 'CAMELS_UY_9',
    'Paso de la Compania': 'CAMELS_UY_1',
    'Tacuarembo': 'CAMELS_UY_4',
    'Durazno': 'CAMELS_UY_13'
}


In [4]:
df = pd.read_csv(attributes_files / "attributes_other.csv")

df = df.merge(
    pd.read_csv(attributes_files / "attributes_caravan.csv"),
    on="gauge_id",
    how="outer"
)

df = df.merge(
    pd.read_csv(attributes_files / "attributes_hydroatlas.csv"),
    on="gauge_id",
    how="outer"
)


In [5]:
# Set gauge_id as the index
attrs_df = df.set_index("gauge_id")

# Select only the basins we want to use
target_gauges = list(basins_mapping.values())
attrs_df = attrs_df.loc[target_gauges]

# Define required attributes
attributes_mapping = {
    'gauge_name': 'gauge_name',
    'ele_mt_sav': 'elev_mean',
    'slp_dg_sav': 'slope_mean',
    'p_mean': 'p_mean',
    'area': 'area_gages2',
    'frac_snow': 'frac_snow',
    'aridity_FAO_PM': 'aridity',
    'pet_mean_FAO_PM': 'pet_mean',
    'high_prec_freq': 'high_prec_freq',
    'high_prec_dur': 'high_prec_dur',
    'low_prec_freq': 'low_prec_freq',
    'low_prec_dur': 'low_prec_dur',
    'snd_pc_sav': 'sand_frac',
    'slt_pc_sav': 'silt_frac',
    'cly_pc_sav': 'clay_frac'
}

attributes_to_keep = {
    'gauge_name',
    'ele_mt_sav',    # Mean elevation (m)
    'slp_dg_sav',   # Catchment slope (m/km)
    'p_mean',       # Mean daily precipitation (mm/day)
    'area',
    'frac_snow',    # Snow fraction (-)
    'aridity_FAO_PM',
    'pet_mean_FAO_PM',
    'high_prec_freq',
    'high_prec_dur',
    'low_prec_freq',
    'low_prec_dur',
    'snd_pc_sav',
    'slt_pc_sav',
    'cly_pc_sav'
}


In [6]:
# Select and process static attributes
final_attrs = attrs_df[list(attributes_to_keep)]
final_attrs = final_attrs.rename(columns=attributes_mapping)

# Check for any missing values in attributes
missing_values = final_attrs.isnull().sum()
if missing_values.any():
    print("\nERROR: Found missing values in attributes:")
    print(missing_values[missing_values > 0])
    print("\nPlease fix the missing values in the source data before proceeding.")
    exit(1)

In [7]:
# Save attributes file
final_attrs.to_csv(attributes_dir / "attributes.csv")
print(f"\nSaved static attributes for {len(final_attrs)} basins")

# Create a standard time index for the period we want
start_date = pd.Timestamp('1989-01-01') #('1989-09-01')  #('1999-10-01')
end_date = pd.Timestamp('2019-12-31') #('2009-08-31')  #('2024-09-30')
date_index = pd.date_range(start=start_date, end=end_date, freq='D', name='date')
print(f"\nUsing time period: {date_index[0]} to {date_index[-1]} ({len(date_index)} days)")

# Process each basin's time series data
print(f"\nProcessing {len(basins)} basins...")

areas= pd.read_csv(area_file)
areas = areas.set_index('gauge_id')

for basin in basins:
    basin_id=basins_mapping[basin]
    print(f"\nProcessing basin {basin}")
    
    # Read meteorological data
    met_file = met_dir / f"{basin_id}.nc"
        
    try:
        # Read meteorological data (xarray Dataset)
        met_ds = xr.open_dataset(met_file)

        # Check for required variables
        required_vars = list(variable_mapping.keys())
        available_vars = set(met_ds.data_vars)
        
        missing_vars = [v for v in required_vars if v not in available_vars]
        if missing_vars:
            print(f"Warning: Missing variables in meteorological data for {basin}: {missing_vars}")
            print("Available variables:", list(available_vars))
            continue
        
        # Select only variables we need
        met_ds = met_ds[required_vars]
        
        # Rename using your mapping
        met_ds = met_ds.rename(variable_mapping)
        
        # Convert xarray → pandas using existing datetime coordinate
        met_df = met_ds.to_dataframe()
    
        # Ensure the index is named "date"
        met_df.index.name = "date"
        
        # Keep only final variables you want
        variables_to_keep = [
            "tmin_C",
            "tmax_C",
            "srad_W_m2",
            "prcp_mm_day"
        ]
        
        met_df = met_df[variables_to_keep]
        
        # Get streamflow data and convert from m³/s to mm/day
        qobs = streamflow_data[basin]    
        area_m2 = areas.loc[basin_id, 'area'] * 1e6  # Convert km² to m²
        
        qobs_mmday = (qobs * 86400 * 1000) / area_m2  # Convert m³/s to mm/day

        # Combine all variables into a single DataFrame
        met_df['QObs_mm_d'] = qobs_mmday
        
        # Reindex to standard date range, filling gaps with NaN
        met_df = met_df.reindex(date_index)
        
        # Convert to xarray Dataset
        ds = met_df.to_xarray()
        
        # Save to netCDF
        encoding = {var: {'_FillValue': np.nan} for var in ds.data_vars}
        encoding['date'] = {
            'dtype': 'float64',
            'calendar': 'proleptic_gregorian',
            'units': f'days since {start_date.strftime("%Y-%m-%d")}'
        }
        
        ds.to_netcdf(
            time_series_dir / f"{basin_id}.nc",
            encoding=encoding
        )
        
        print(f"Saved time series with {len(ds.data_vars)} variables")
        
    except FileNotFoundError:
        print(f"Warning: No meteorological data found for basin {basin_id}")
        continue
    except Exception as e:
        print(f"Error processing basin {basin_id}: {str(e)}")
        continue

print("\nData preparation complete!")



Saved static attributes for 16 basins

Using time period: 1989-01-01 00:00:00 to 2019-12-31 00:00:00 (11322 days)

Processing 11 basins...

Processing basin Paso de las Toscas
Saved time series with 5 variables

Processing basin Paso Aguiar
Saved time series with 5 variables

Processing basin Paso de las Piedras
Saved time series with 5 variables

Processing basin Paso de las Piedras (R3)
Saved time series with 5 variables

Processing basin Paso Mazangano
Saved time series with 5 variables

Processing basin Fraile Muerto
Saved time series with 5 variables

Processing basin Paso del Borracho
Saved time series with 5 variables

Processing basin Paso Baltasar
Saved time series with 5 variables

Processing basin Paso de Coelho
Saved time series with 5 variables

Processing basin Paso Manuel Diaz
Saved time series with 5 variables

Processing basin Bequelo
Saved time series with 5 variables

Data preparation complete!
